In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "ftfy", "ydata-profiling", "openpyxl", "ipython", "-q"])

0

# Detección de Anorexia con Machine Learning en Redes Sociales
La detección automática de trastornos de la alimentación (TA) en plataformas de redes sociales constituye el enfoque central de esta revisión de vanguardia. Mediante el análisis de investigaciones especializadas, se identifican tres arquitecturas representativas: la Red de Atención Jerárquica para la Detección de la Depresión Multiaspecto (MDHAN), aplicada a datos de Twitter; el modelo de aprendizaje por transferencia basado en BETO para la detección de la anorexia en español; y la Unidad Multimodal con Compuerta Multicanal (GMU) para la caracterización holística del usuario. La evaluación técnica destaca a MDHAN como la arquitectura con la mayor precisión reportada (99,86 %), lograda mediante la optimización híbrida de enjambre de partículas y lobo gris (PWO) para la selección de características. Se documenta la evolución metodológica, desde los modelos clásicos basados ​​en SVM hasta las soluciones actuales basadas en Transformers, con énfasis en los biomarcadores lingüísticos para el apoyo a la toma de decisiones clínicas en el ámbito específico de los trastornos de la alimentación. Se presenta un análisis de compensación operativa que compara la precisión, la complejidad computacional y la interpretabilidad entre las distintas arquitecturas, junto con un conjunto de objetivos experimentales propuestos para futuros trabajos.

## Descripción del modelo de solución propuesto
El análisis del estado del arte en la detección de trastornos mentales revela una trayectoria que comenzó con el uso intensivo de Bag of Words y SVM, modelos que, si bien son eficientes en términos computacionales, carecen de la capacidad para representar el contexto bidireccional necesario para entender la salud mental en redes sociales. La limitación fundamental de la métrica TF-IDF radica en su naturaleza estática; al otorgar peso a las palabras en función de su frecuencia inversa en el corpus, se ignoran las relaciones semánticas entre términos que, aunque no compartan la misma raíz, expresan la misma intención de riesgo. Por ejemplo, en el dominio de los TCA, términos como "restricción", "ayuno" o "control" pueden aparecer en contextos totalmente ajenos a la patología si no se analiza la estructura semántica de la oración.

Al eliminar el uso de SVM y TF-IDF, la arquitectura propuesta se libera de la dependencia de la frecuencia léxica. En su lugar, se propone el uso de vectores densos generados por el codificador de texto de MobileCLIP. Esta tecnología, desarrollada para optimizar el rendimiento en tiempo real en dispositivos móviles, utiliza una arquitectura híbrida de Transformer y redes convolucionales con reparametrización estructural, lo que le permite capturar matices semánticos complejos con una latencia significativamente inferior a la de los Transformers convencionales. La elección de MobileCLIP sobre el ajuste fino de BETO es estratégica: mientras que BETO se especializa en el léxico en español, MobileCLIP captura la intención semántica global mediante un espacio de embebido alineado con conceptos visuales y lingüísticos universales, lo que resulta especialmente útil cuando se cuenta con un dataset limitado de 1,500 tweets donde la diversidad lingüística es alta pero el volumen de datos es bajo.

## Objetivo de la solución
Como lo vimos en el anterior punto, los modelos tradicionales usados para la detección de TCAs o diferentes trastornos mentales, se quedan cortos en la interpretación del contexto. Aunado a esto, el lenguaje coloquial e intrincado que pueden tener las redes sociales dificulta más esta tarea.

Por lo anterior, se ha decidido dejar de lado el enfoque tradicional de usar SVM e IT-IDF por una capa de abstracción semántica con la técnica de NLI. La anterior arquitectura tiene como ventaja que no es entrenado, sino que se utiliza en un modo de inferencia "zero-shot" para evaluar si un tweet determinado implica una hipótesis de riesgo específica. Este enfoque transforma la clasificación binaria en un problema de evaluación de premisas. Se definen una serie de etiquetas de riesgo clínico basadas en biomarcadores lingüísticos documentados, y el modelo calcula la probabilidad de que el texto del tweet sea una evidencia de dicha etiqueta.

A continuación utlizaremos la librería `ydata-profiling`la cual es una librería de Python que automatiza el análisis exploratorio de datos (EDA), generando reportes detallados, interactivos y visuales a partir de un DataFrame de pandas con una sola línea de código.

## Explicación del dataset
El dataset contiene datos codificados erroneamente en otro encoder diferente al estándar UTF-8. Debido a lo anterior, se genera un problema de interpretación de caracteres que pude meter ruido al modelo y por consiguiente entregarnos resultados sesgados e incorrectos. 

Para evitar el problema planteado, se aplicará un preprocesamiento de datos para convertir los caracteres especiales al encoder UTF-8 y capturar de manera eficiente la semántica de los textos sin que contengan ruido.

## Procesamiento de dataset y liempieza de ruido
Para realizar este procesamiento y eliminar los caracteres especiales teدثmos dos posibles librerías: `ftfy` y `clean-text`. 
- ftfy (Fix Text For You): Es la herramienta estándar para arreglar el "Mojibake" (caracteres corruptos por errores de codificación). El dataset tiene símbolos extraños donde debería haber tildes o eñes, por lo tanto, ftfy es indispensable.

- clean-text: Es más una multi-herramienta para normalizar. Sirve para quitar URLs, correos, convertir a minúsculas y eliminar emojis de forma masiva.

En conclusión, ftfy es superior para la fase inicial de rescate del texto. Luego, usaremos funciones REGEX (Expresiones Regulares) para el ruido, ya que, clean-text a veces es demasiado agresivo y podría eliminar signos de puntuación que pueden tener valor clínico.

Ahora, otro dilema importante es si quitar las "stopwords"(palabras como artículos y conectores que no aportan valor semántico) de los tweets. Sin embargo basandonos en el estudio: Understanding Patterns of Anorexia Manifestations in Social Media Data with Deep Learning - ACL Anthology. Este menciona que los resultados revelaron patrones psicológicos en los usuarios con anorexia, gracias al  análisis de las capas internas del modelo se identificó una clara tendencia al aislamiento social, reflejada en un menor uso de pronombres inclusivos (como "nosotros") frente a los individuales. Un hallazgo relevante fue el abandono de lo que ellos llamaron "actividad explicativa": los usuarios con riesgo tienden a usar menos conectores causales (como "porque" o "debido a"), lo cual los autores relacionan con un estado psicológico de indefensión aprendida y desesperanza, donde el usuario deja de intentar explicar o entender su propia situación.

Para finalizar el análisis previo, al usar Random Forest, el modelo puede detectar si la frecuencia de estas palabras "funcionales" tiene peso predictivo. Si las borramos en este proceso tan temprano, corremos el riesgo de perder estas señales.


In [2]:
import pandas as pd
import ftfy
import re
from ydata_profiling import ProfileReport
from IPython.display import Image

# 1. Configuración de rutas
input_path = '/Users/ignacio/Documents/Proyecto Apps/Reto_App_Avanzadas/data_train.xlsx' 
output_path = '/Users/ignacio/Documents/Proyecto Apps/Reto_App_Avanzadas/data_train_reparado.csv'

def preprocesamiento(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return ""
    
    # 1.1 Rescate de caracteres (Mojibake)
    # Para no perder eñes o tildes que cambian el significado
    texto = ftfy.fix_text(texto)
    
    # 1.2 Normalización de ruido externo
    # Eliminamos menciones (@user) y URLs porque no contienen carga semántica del usuario
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto) # Quitar URLs
    texto = re.sub(r'@\w+', '', texto)                  # Quitar menciones
    
    # 1.3 Limpieza de caracteres especiales preservando puntuación
    # Mantenemos letras, números, espacios y signos de puntuación básicos (. , ! ?)
    # Los puntos suspensivos o exclamaciones pueden ser biomarcadores de estado de ánimo
    texto = re.sub(r'[^\w\s.,!?¡¿]', '', texto) 
    
    # 1.4 Manejo de espacios y saltos
    texto = texto.replace('\n', ' ').replace('\r', '')
    texto = re.sub(r'\s+', ' ', texto) # Eliminar espacios dobles
    
    return texto.strip().lower() # Se convierte a minúsculas para tener uniformidad en RL y RF

df = pd.read_excel(input_path)

# 2. Aplicamos la limpieza
df['tweet_text_clean'] = df['tweet_text'].apply(preprocesamiento)

# 3. Guardar solo el texto limpio y la etiqueta en utf-8
df.drop(columns=["tweet_text"]).to_csv(output_path, index=False, encoding="utf-8")

print(f"Dataset limpio guardado. Total registros: {len(df)}")
print(df[['tweet_text', 'tweet_text_clean']].head())

# 4. Análisis exploratorio con ydata-profiling
# Generar el reporte centrado en el texto limpio y las etiquetas
profile = ProfileReport(df, title="Reporte de Perfilado - Detección Anorexia", explorative=True)

# Guarda el reporte en un archivo HTML para visualizarlo en el navegador
profile.to_file("reporte_analisis_tweets.html")
print("Reporte de análisis generado revisa los archivos en el directorio actual para encontrarlo.")

/Users/ignacio/Documents/apuntes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/41/qv7vy6q53nz7dkqkrsww7qm80000gn/T/ipykernel_91868/2112586569.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Dataset limpio guardado. Total registros: 1500
                                          tweet_text  \
0  Cheesecake saludable sin azÃºcar y sin lactosa...   
1           ser como ellas â™¡â™¡\n  #HastaLosHuesos   
2  Comida Real o , la clave para estar mÃ¡s sana,...   
3  Entre el cambio de hora y la bajada de las #te...   
4   Hace mucho tiempo no sentÃ­a mi cuerpo tan frÃ­o   

                                    tweet_text_clean  
0  cheesecake saludable sin azúcar y sin lactosa ...  
1                      ser como ellas hastaloshuesos  
2  comida real o , la clave para estar más sana, ...  
3  entre el cambio de hora y la bajada de las tem...  
4     hace mucho tiempo no sentía mi cuerpo tan frío  


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/10 [00:00<?, ?it/s, Describe variable: user_id]

...

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 18.45it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 1167.03it/s]

Reporte de análisis generado revisa los archivos en el directorio actual para encontrarlo.


In [3]:
try:
    Image(filename='/Users/yaelpm.dev08/Desktop/Reto_App_Avanzadas/desbalance.png')
except FileNotFoundError:
    print("Imagen no disponible en esta máquina")

Imagen no disponible en esta máquina


In [4]:
try:
    Image(filename='/Users/yaelpm.dev08/Desktop/Reto_App_Avanzadas/palabras.png')
except FileNotFoundError:
    print("Imagen no disponible en esta máquina")

Imagen no disponible en esta máquina


### Análisis exploratorio de los datos

NOTA: Para ver el archivo `reporte_analisis_tweets.html` es recometable abrirlo con el navegador o con la extensión "Live Server" de Visual Studio Code

Como podemos ver en la imagen con la ayuda de esta librería pudimos observar que existe un desbalance de clases entre "anorexia" y "control". Tenemos que tomar en cuenta lo anterior al momento de hacer la división para el entrenamiento y la validación.

Si observamos la segunda imágen esta contiene un lista de palabras repetidas en los tweets, la mayoría de ellas son stopwords, recordemos que hasta este punto no hemos quitado estas palabras para no perder ciertas relaciones contextuales. Pero es de especial importancia notar que existen palabras relacionadas con la anorexia que se repiten, se listan a continuación:

- anorexia
- bulimia
- comer
- hambre
- cuerpo
- comer

### Normalización de slangs 

En el reto una fase sugerida es la normalización de abreviaturas que se refiere a expandir abreviaturas comunes en redes sociales ("u" → "you", "pls" → "please"). Esto debido a que los modelos pueden interpretar significados diferentes según la escritura aunque la palabra tenga la misma conotación.